In [ ]:
# 1) Setup + data unzip
from google.colab import drive

drive.mount('/content/drive')

%pip install -q -U torch torchvision transformers pandas pillow tqdm accelerate qwen-vl-utils "bitsandbytes>=0.46.1" peft

import ast
import os
import random
import re
import zipfile

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset, Subset
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2VLForConditionalGeneration, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

ZIP_PATH = "/content/drive/MyDrive/SNU_AI_Challenge/snuaichallenge.zip"
DATA_DIR = "/content/snuaichallenge_data"
OUTPUT_DIR = "/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_multitask_v1"
SUBMIT_PATH = "/content/outputs/submission.csv"

if not os.path.isdir(DATA_DIR):
    with zipfile.ZipFile(ZIP_PATH) as zip_file:
        zip_file.extractall("/content/")

TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TEST_CSV = os.path.join(DATA_DIR, "test.csv")
TRAIN_IMAGE_DIR = os.path.join(DATA_DIR, "train")
TEST_IMAGE_DIR = os.path.join(DATA_DIR, "test")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.dirname(SUBMIT_PATH), exist_ok=True)

assert os.path.exists(TRAIN_CSV), TRAIN_CSV
assert os.path.exists(TEST_CSV), TEST_CSV
assert os.path.isdir(TRAIN_IMAGE_DIR), TRAIN_IMAGE_DIR
assert os.path.isdir(TEST_IMAGE_DIR), TEST_IMAGE_DIR

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
MIN_PIXELS = 128 * 28 * 28
MAX_PIXELS = 256 * 28 * 28
SEED = 42
VALID_RATIO = 0.1
SMOKE_TEST = False

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("device:", "cuda" if torch.cuda.is_available() else "cpu")
print("data:", DATA_DIR)
print("output:", OUTPUT_DIR)


In [ ]:
# 2) Dataset, collator, prediction helpers
def parse_answer(answer):
    result = answer if isinstance(answer, list) else ast.literal_eval(str(answer))
    result = [int(value) for value in result]
    if len(result) != 4 or sorted(result) != [1, 2, 3, 4]:
        raise ValueError(f"Invalid Answer: {answer}")
    return result


train_df = pd.read_csv(TRAIN_CSV)
train_df["Id"] = train_df["Id"].astype(str)
train_df["Answer_list"] = train_df["Answer"].apply(parse_answer)

test_df = pd.read_csv(TEST_CSV)
test_df["Id"] = test_df["Id"].astype(str)

unique_ids = train_df["Id"].unique().copy()
rng = np.random.default_rng(SEED)
rng.shuffle(unique_ids)
valid_size = max(1, int(len(unique_ids) * VALID_RATIO))
valid_ids = set(unique_ids[:valid_size])
training_ids = set(unique_ids[valid_size:])

training_df = train_df[train_df["Id"].isin(training_ids)].reset_index(drop=True)
validation_df = train_df[train_df["Id"].isin(valid_ids)].reset_index(drop=True)


class FrameOrderDataset(Dataset):
    def __init__(self, dataframe, image_root, augment=True, seed=42):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_root = image_root
        self.augment = augment
        self.seed = seed

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        sample_id = str(row["Id"])
        sentence = "" if pd.isna(row["Sentence"]) else str(row["Sentence"])

        image_paths = [
            os.path.join(self.image_root, sample_id, str(row[f"Input_{i}"]))
            for i in range(1, 5)
        ]
        temporal_ranks = [int(value) for value in row["Answer_list"]]

        local_rng = random if self.augment else random.Random(self.seed + index)
        if self.augment:
            permutation = list(range(4))
            local_rng.shuffle(permutation)
            image_paths = [image_paths[i] for i in permutation]
            temporal_ranks = [temporal_ranks[i] for i in permutation]

        instruction = (
            f'The video is described as: "{sentence}"\n\n'
            "The four images are shuffled frames from the video. "
            "Compare the visual states and temporal progression.\n"
            "Return the actual temporal rank of Input image 1, "
            "Input image 2, Input image 3, and Input image 4 "
            "in that order.\n"
            "Respond only with one Python-style permutation list."
        )

        return {
            "Id": sample_id,
            "image_paths": image_paths,
            "image_labels": [f"Input image {i}" for i in range(1, 5)],
            "instruction": instruction,
            "target": str(temporal_ranks),
        }


training_dataset = FrameOrderDataset(training_df, TRAIN_IMAGE_DIR, augment=True, seed=SEED)
validation_dataset = FrameOrderDataset(validation_df, TRAIN_IMAGE_DIR, augment=False, seed=SEED)


def load_rgb(path):
    with Image.open(path) as image:
        return image.convert("RGB").copy()


def make_messages(example):
    content = []
    for label in example["image_labels"]:
        content.append({"type": "text", "text": f"\n{label}:"})
        content.append({"type": "image"})
    content.append({"type": "text", "text": "\n\n" + example["instruction"]})
    return [{"role": "user", "content": content}]


def find_last_subsequence(sequence, pattern):
    for start in range(len(sequence) - len(pattern), -1, -1):
        if sequence[start:start + len(pattern)] == pattern:
            return start
    return -1


class QwenFrameOrderCollator:
    def __init__(self, processor):
        self.processor = processor
        self.assistant_prefix_ids = processor.tokenizer.encode(
            "<|im_start|>assistant\n",
            add_special_tokens=False,
        )

    def __call__(self, examples):
        if len(examples) != 1:
            raise ValueError("Use batch size 1 with this collator.")

        example = examples[0]
        images = [load_rgb(path) for path in example["image_paths"]]
        messages = make_messages(example) + [{"role": "assistant", "content": example["target"]}]
        text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

        model_inputs = self.processor(text=[text], images=images, padding=False, return_tensors="pt")
        input_ids = model_inputs["input_ids"][0].tolist()
        assistant_pos = find_last_subsequence(input_ids, self.assistant_prefix_ids)

        if assistant_pos >= 0:
            answer_start = assistant_pos + len(self.assistant_prefix_ids)
        else:
            prompt_text = self.processor.apply_chat_template(make_messages(example), tokenize=False, add_generation_prompt=True)
            prompt_inputs = self.processor(text=[prompt_text], images=images, padding=False, return_tensors="pt")
            answer_start = prompt_inputs["input_ids"].shape[1]

        labels = model_inputs["input_ids"].clone()
        labels[:, :answer_start] = -100
        if "attention_mask" in model_inputs:
            labels[model_inputs["attention_mask"] == 0] = -100
        model_inputs["labels"] = labels
        return model_inputs


def parse_model_output(output_text):
    match = re.search(r"\[\s*[1-4]\s*,\s*[1-4]\s*,\s*[1-4]\s*,\s*[1-4]\s*\]", output_text)
    if match is None:
        return None
    try:
        result = ast.literal_eval(match.group())
    except (ValueError, SyntaxError, TypeError):
        return None
    if isinstance(result, list) and len(result) == 4 and sorted(result) == [1, 2, 3, 4]:
        return [int(value) for value in result]
    return None


@torch.no_grad()
def predict_example(example, max_new_tokens=32):
    model.eval()
    images = [load_rgb(path) for path in example["image_paths"]]
    text = processor.apply_chat_template(make_messages(example), tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=images, padding=False, return_tensors="pt").to(model.device)
    generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
    output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    return output_text, parse_model_output(output_text)

print("train rows:", len(training_dataset), "validation rows:", len(validation_dataset), "test rows:", len(test_df))


In [ ]:
# 3) Train
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.config.use_cache = False

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(
    model,
    LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        target_modules=["q_proj", "v_proj"],
        task_type="CAUSAL_LM",
    ),
)

for name, parameter in model.named_parameters():
    if "visual" in name.lower() or "vision" in name.lower():
        parameter.requires_grad = False

model.print_trainable_parameters()

data_collator = QwenFrameOrderCollator(processor)

if SMOKE_TEST:
    train_data = Subset(training_dataset, range(min(30, len(training_dataset))))
    eval_data = Subset(validation_dataset, range(min(10, len(validation_dataset))))
    max_steps = 100
    eval_steps = 5
else:
    train_data = training_dataset
    eval_data = validation_dataset
    max_steps = -1
    eval_steps = 100

training_argument_values = {
    "output_dir": OUTPUT_DIR,
    "num_train_epochs": 1,
    "max_steps": max_steps,
    "per_device_train_batch_size": 1,
    "per_device_eval_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "learning_rate": 1e-4,
    "warmup_ratio": 0.03,
    "max_grad_norm": 0.3,
    "fp16": True,
    "bf16": False,
    "gradient_checkpointing": True,
    "gradient_checkpointing_kwargs": {"use_reentrant": False},
    "optim": "paged_adamw_8bit",
    "eval_steps": eval_steps,
    "logging_steps": 20,
    "save_strategy": "steps",
    "save_steps": 100,
    "save_total_limit": "None",
    "report_to": "none",
    "remove_unused_columns": False,
    "dataloader_num_workers": 0,
    "seed": SEED,
    "data_seed": SEED,
}

if "eval_strategy" in TrainingArguments.__init__.__code__.co_varnames:
    training_argument_values["eval_strategy"] = "steps"
else:
    training_argument_values["evaluation_strategy"] = "steps"

training_args = TrainingArguments(**training_argument_values)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=eval_data,
    data_collator=data_collator,
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)


In [ ]:
# 4) Eval
model.eval()
model.config.use_cache = True

validation_results = []
for index in tqdm(range(len(validation_dataset)), desc="Validation"):
    example = validation_dataset[index]
    raw_output, prediction = predict_example(example)
    answer = ast.literal_eval(example["target"])
    answer = [int(value) for value in answer]
    validation_results.append({
        "Id": example["Id"],
        "Raw_output": raw_output,
        "Prediction": str(prediction) if prediction is not None else None,
        "Answer": str(answer),
        "Correct": prediction == answer,
    })

validation_result_df = pd.DataFrame(validation_results)
exact_match_accuracy = validation_result_df["Correct"].mean()
parse_failure_count = validation_result_df["Prediction"].isna().sum()

print(f"Exact-match accuracy: {exact_match_accuracy:.2%}")
print(f"Correct: {validation_result_df['Correct'].sum()}/{len(validation_result_df)}")
print(f"Parse failures: {parse_failure_count}")
display(validation_result_df.head())


In [ ]:
# 5) Inference + submission
predictions = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inference"):
    sentence = "" if pd.isna(row["Sentence"]) else str(row["Sentence"])
    image_paths = [
        os.path.join(TEST_IMAGE_DIR, str(row["Id"]), str(row[f"Input_{i}"]))
        for i in range(1, 5)
    ]
    example = {
        "Id": str(row["Id"]),
        "image_paths": image_paths,
        "image_labels": [f"Input image {i}" for i in range(1, 5)],
        "instruction": (
            f'The video is described as: "{sentence}"\n\n'
            "The four images are shuffled frames from the video. "
            "Compare the visual states and temporal progression.\n"
            "Return the actual temporal rank of Input image 1, "
            "Input image 2, Input image 3, and Input image 4 "
            "in that order.\n"
            "Respond only with one Python-style permutation list."
        ),
    }

    _, prediction = predict_example(example)
    predictions.append({
        "Id": str(row["Id"]),
        "Answer": str(prediction if prediction is not None else [1, 2, 3, 4]),
    })

submission_df = pd.DataFrame(predictions)
submission_df.to_csv(SUBMIT_PATH, index=False)
print("saved:", SUBMIT_PATH)
display(submission_df.head())
